# FinTrust Digital Bank — Week 2: Build & Analyse
## AnalystLab Africa Experience Lab — Week 2

**Scope note:** the official Week 2 brief was not provided. This notebook follows (a) the Week 1 brief,
which states Week 2 introduces the requirement to *exchange and use tangible outputs from other tracks*,
and (b) the Week 2–4 plan committed to in the Week 1 report.

**Data disclaimer:** all FinTrust data is synthetic and created for educational purposes.


## 1. Setup

In [ ]:
import pandas as pd, numpy as np, json
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import StandardScaler

cust = pd.read_excel("FinTrust_Customer_Data.xlsx")
txn  = pd.read_excel("FinTrust_Transaction_Data.xlsx")
m = txn.merge(cust, on="Customer_ID", how="left")
print("merged:", m.shape)


## 2. Data preparation
Impute the 96 missing categorical cells, keep Location and City separate, add a log amount.

In [ ]:
m["Hour"] = m.Transaction_DateTime.dt.hour
m["DayOfWeek"] = m.Transaction_DateTime.dt.dayofweek
m["IsWeekend"] = (m.DayOfWeek >= 5).astype(int)
m["IsNight"] = m.Hour.isin([0,1,2,3,4,5]).astype(int)
m["IsIntl"] = (m.International_Transaction == "Yes").astype(int)
m["LogAmount"] = np.log1p(m.Amount_NGN)
m["HighValue"] = (m.Amount_NGN > m.Amount_NGN.quantile(0.90)).astype(int)
m["Risk_Yes"] = (m.Risk_Review_Flag == "Yes").astype(int)
print("missing after prep:", m.isna().sum().sum())


## 3. Customer-level aggregates (feature engineering)

In [ ]:
agg = txn.groupby("Customer_ID").agg(
    Cust_TxnCount=("Transaction_ID","size"),
    Cust_AvgAmount=("Amount_NGN","mean"),
    Cust_TotalValue=("Amount_NGN","sum"),
    Cust_FailRate=("Transaction_Status", lambda s: s.isin(["Failed","Reversed"]).mean()),
    Cust_IntlRate=("International_Transaction", lambda s: (s=="Yes").mean()),
    Cust_NightRate=("Transaction_DateTime", lambda s: s.dt.hour.isin([0,1,2,3,4,5]).mean()),
).reset_index()
m = m.merge(agg, on="Customer_ID", how="left")
m["AmountVsCustAvg"] = m.Amount_NGN / m.Cust_AvgAmount.replace(0, np.nan)
print("engineered columns:", m.shape[1])


## 4. Engineered-feature significance tests

In [ ]:
for col in ["IsIntl","IsNight","IsWeekend","HighValue"]:
    x2, p, _, _ = stats.chi2_contingency(pd.crosstab(m[col], m.Risk_Review_Flag))
    print(f"{col:11s} chi2={x2:8.2f} p={p:.3e} {'SIGNIFICANT' if p<0.05 else 'not sig'}")

for col in ["LogAmount","Cust_AvgAmount","Cust_TxnCount","AmountVsCustAvg","Cust_FailRate","Cust_IntlRate","Cust_NightRate"]:
    a = m[col][m.Risk_Yes==1]; b = m[col][m.Risk_Yes==0]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    print(f"{col:16s} t={t:8.2f} p={p:.3e} {'SIGNIFICANT' if p<0.05 else 'not sig'}")

print("\nlift analysis:")
print(m.groupby("IsIntl").Risk_Yes.mean().round(4).to_string())
print(m.groupby("IsNight").Risk_Yes.mean().round(4).to_string())


## 5. Analytical SQL pack
Ten queries executed through pandas equivalents. Full SQL: `fintrust_week2_queries.sql`.

In [ ]:
q1 = (m.groupby(m.Transaction_DateTime.dt.to_period("M"))
        .agg(transactions=("Transaction_ID","size"), value_NGN=("Amount_NGN","sum"),
             success_rate=("Transaction_Status", lambda s: round(100*(s=="Successful").mean(),2)))
        .reset_index())
q2 = (m.groupby("Channel")
        .agg(transactions=("Transaction_ID","size"), value_NGN=("Amount_NGN","sum"),
             failed=("Transaction_Status", lambda s: int((s=="Failed").sum())),
             risk_rate=("Risk_Review_Flag", lambda s: round(100*(s=="Yes").mean(),2)))
        .reset_index().sort_values("risk_rate", ascending=False))
q3 = (m.groupby("Transaction_Type")
        .agg(transactions=("Transaction_ID","size"), total_value=("Amount_NGN","sum"),
             avg_value=("Amount_NGN","mean"),
             risk_rate=("Risk_Review_Flag", lambda s: round(100*(s=="Yes").mean(),2)))
        .reset_index().sort_values("risk_rate", ascending=False))
q4 = (m.groupby("Customer_Segment")
        .agg(transactions=("Transaction_ID","size"), customers=("Customer_ID","nunique"),
             value_NGN=("Amount_NGN","sum"),
             risk_rate=("Risk_Review_Flag", lambda s: round(100*(s=="Yes").mean(),2)))
        .reset_index().sort_values("value_NGN", ascending=False))
q5 = (m.groupby("Hour")
        .agg(transactions=("Transaction_ID","size"),
             risk_rate=("Risk_Review_Flag", lambda s: round(100*(s=="Yes").mean(),2)))
        .reset_index())
q6 = (m.groupby(["Customer_ID","Customer_Name","Customer_Segment"])
        .agg(transactions=("Transaction_ID","size"), total_value=("Amount_NGN","sum"))
        .reset_index().sort_values("total_value", ascending=False).head(20))
q7 = (m[m.Risk_Yes==1].groupby(["Customer_ID","Customer_Name"])
        .agg(flagged_txns=("Transaction_ID","size"), flagged_value=("Amount_NGN","sum"))
        .reset_index().sort_values("flagged_value", ascending=False).head(20))
q8 = m.pivot_table(index=m.Transaction_DateTime.dt.to_period("M").astype(str),
                   columns="Transaction_Status", values="Transaction_ID",
                   aggfunc="count", fill_value=0).reset_index()
q9 = (m.groupby("Location")
        .agg(transactions=("Transaction_ID","size"), value_NGN=("Amount_NGN","sum"),
             risk_rate=("Risk_Review_Flag", lambda s: round(100*(s=="Yes").mean(),2)))
        .reset_index().sort_values("value_NGN", ascending=False))
q10 = pd.DataFrame({
    "feature": ["IsIntl","IsNight","IsWeekend","HighValue"],
    "risk_rate_yes": [round(100*m[m[c]==1].Risk_Yes.mean(),2) for c in ["IsIntl","IsNight","IsWeekend","HighValue"]],
    "risk_rate_no":  [round(100*m[m[c]==0].Risk_Yes.mean(),2) for c in ["IsIntl","IsNight","IsWeekend","HighValue"]],
    "lift": [round(m[m[c]==1].Risk_Yes.mean()/m[m[c]==0].Risk_Yes.mean(),2) for c in ["IsIntl","IsNight","IsWeekend","HighValue"]]})

for name, df in [("Q1",q1),("Q2",q2),("Q3",q3),("Q4",q4),("Q5",q5),("Q6",q6),("Q7",q7),("Q8",q8),("Q9",q9),("Q10",q10)]:
    print("="*70); print(name); print(df.to_string(index=False))

q10.to_csv("Q10_feature_risk_rates.csv", index=False)


## 6. Model improvement over the Week 1 baseline
Week 1: ROC-AUC 0.673 but recall 0.039 at the default 0.50 threshold. Week 2 applies class balancing, sample weighting and threshold tuning.

In [ ]:
num_feats = ["Amount_NGN","LogAmount","IsIntl","IsNight","IsWeekend","HighValue","Age",
             "Tenure_Months","Digital_Engagement_Score","Hour","Cust_TxnCount","Cust_AvgAmount",
             "AmountVsCustAvg","Cust_FailRate","Cust_IntlRate","Cust_NightRate"]
cat_feats = ["Transaction_Type","Channel","Device_Type","Transaction_Status",
             "Customer_Segment","Account_Status","Account_Type","Monthly_Income_Band"]
X = m[num_feats + cat_feats].copy()
X[cat_feats] = X[cat_feats].fillna("Unknown")
X = pd.get_dummies(X, columns=cat_feats, drop_first=True).fillna(0).astype(float)
y = m.Risk_Yes
print("feature matrix:", X.shape, "| positive rate %.2f%%" % (100*y.mean()))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
scale_cols = ["Amount_NGN","LogAmount","Age","Tenure_Months","Digital_Engagement_Score",
              "Cust_TxnCount","Cust_AvgAmount","AmountVsCustAvg"]
sc = StandardScaler().fit(Xtr[scale_cols])
Xtr_s, Xte_s = Xtr.copy(), Xte.copy()
Xtr_s[scale_cols] = sc.transform(Xtr[scale_cols]); Xte_s[scale_cols] = sc.transform(Xte[scale_cols])

def report(name, proba):
    auc = roc_auc_score(yte, proba); ap = average_precision_score(yte, proba)
    prec, rec, thr = precision_recall_curve(yte, proba)
    f1 = 2*prec*rec/np.maximum(prec+rec, 1e-9); bi = int(np.nanargmax(f1[:-1])); th = thr[bi]
    pred = (proba >= th).astype(int)
    order = np.argsort(-proba)
    r10 = yte.values[order[:int(.10*len(yte))]].mean()/yte.mean()
    print("="*70); print(name)
    print("ROC-AUC %.4f | PR-AUC %.4f | base rate %.4f" % (auc, ap, yte.mean()))
    print("best threshold %.3f | precision %.4f | recall %.4f | F1 %.4f" % (th, prec[bi], rec[bi], f1[bi]))
    print("lift @ top 10%% reviewed: %.2fx" % r10)
    print(confusion_matrix(yte, pred))
    print(classification_report(yte, pred, digits=3))

report("LogisticRegression (balanced)",
       LogisticRegression(max_iter=3000, class_weight="balanced").fit(Xtr_s, ytr).predict_proba(Xte_s)[:,1])

sw = np.where(ytr==1, (ytr==0).sum()/(ytr==1).sum(), 1.0)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42).fit(Xtr, ytr, sample_weight=sw)
report("GradientBoosting (weighted)", gb.predict_proba(Xte)[:,1])

cv = cross_val_score(GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
                     X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")
print("\n5-fold CV ROC-AUC:", np.round(cv,4), "| mean %.4f" % cv.mean())
print("\nTop feature importances:")
print(pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False).head(15).round(4).to_string())


## 7. Cross-track exchange
Week 2 requires tangible outputs to be exchanged. See `FinTrust_Week2_CrossTrack_Exchange.md` for the full handover pack (ML Engineering: feature matrix + threshold; GenAI: grounding facts + guardrails; Project Management: dependency and risk update).

## 8. Plan for Weeks 3–4
**Week 3:** serve the model via FastAPI `/predict` with schema validation and tests; run the GenAI retrieval evaluation; confirm the frozen feature contract. **Week 4:** integrate, document, present and submit.